## Imports


In [ ]:
from datasets import load_dataset
from preprocess import preprocess
from NaiveBayes.naivebayes import NaiveBayes
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score , precision_score , recall_score, f1_score , confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

## Load Dataset


In [ ]:
dataset = load_dataset("ag_news")

# shuffle the dataset to ensure that the train and test sets are not ordered in any way
dataset = dataset.shuffle(seed=42)

# split the dataset into train and test sets
train_data = dataset["train"]
test_data = dataset["test"]

print(f"Train size: {len(train_data)}")
print(f"Test size: {len(test_data)}")

print(dataset["test"]["text"][:3])   # first 3 texts
print(dataset["train"][1])
print(dataset["train"][1]["text"])

dataset["train"].to_pandas().head(10)

## Preprocess Data 

In [ ]:
x_train = [preprocess(x) for x in train_data["text"]]
y_train = train_data["label"]

x_test = [preprocess(x) for x in test_data["text"]]
y_test = test_data["label"]

print(x_train[:3])
print(y_train[:3])


# Train the Manual Naive Bayes

In [ ]:
model = NaiveBayes()
model.fit(x_train, y_train)

preds = model.predict(x_test)

## Evaluation for the Manual Model

In [ ]:
# Evaluation Metrics
manual_accuracy = accuracy_score(y_test, preds) * 100   
manual_precision = precision_score(y_test, preds, average='macro') * 100
manual_recall = recall_score(y_test, preds, average='macro') * 100
manual_f1 = f1_score(y_test, preds, average='macro') * 100

print(f"Scratch NB Accuracy: {manual_accuracy:.3f}")
print(f"Scratch NB Precision: {manual_precision:.3f}")
print(f"Scratch NB Recall: {manual_recall:.3f}")
print(f"Scratch NB F1: {manual_f1:.3f}")

# Confusion Matrix
labels = ["World", "Sports", "Business", "Sci/Tech"]
cm_manual = confusion_matrix(y_test, preds)

sns.heatmap(
    cm_manual,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Naive Bayes (Scratch)')
plt.show()
plt.show()  


## SKlearn Model


In [ ]:
vectorizer = CountVectorizer(lowercase=True, stop_words=None)

X_train_vec = vectorizer.fit_transform(train_data["text"])
X_test_vec = vectorizer.transform(test_data["text"])

clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

sklearn_preds = clf.predict(X_test_vec)

## Evaluation for the SKLearn Model

In [ ]:
# Evaluation Metrics
sklearn_accuracy = accuracy_score(y_test, sklearn_preds) * 100
sklearn_precision = precision_score(y_test, sklearn_preds, average='macro') * 100   
sklearn_recall = recall_score(y_test, sklearn_preds, average='macro') * 100
sklearn_f1 = f1_score(y_test, sklearn_preds, average='macro') * 100

print(f"Sklearn NB Accuracy: {sklearn_accuracy:.3f}")
print(f"Sklearn NB Precision: {sklearn_precision:.3f}")
print(f"Sklearn NB Recall: {sklearn_recall:.3f}")
print(f"Sklearn NB F1: {sklearn_f1:.3f}")

# Confusion Matrix
cm_sklearn = confusion_matrix(y_test, sklearn_preds)
sns.heatmap(
    cm_sklearn,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=labels,
    yticklabels=labels
)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix - Naive Bayes (Sklearn)')
plt.show()
